### 환경 설정 및 라이브러리 설치

In [ ]:
# 필수 라이브러리 설치
!pip install --upgrade pip
!pip install torch torchvision torchaudio diffusers transformers accelerate xformers --upgrade --quiet

import torch
import numpy as np
import matplotlib.pyplot as plt
import math
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video # GIF 대신 비디오/이미지 저장을 위해 사용

# GPU 및 데이터 타입 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

# 모델 로드 (v1-4 또는 v1-5 선택 가능)
model_id = "CompVis/stable-diffusion-v1-4"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch_dtype)
pipe = pipe.to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.1.1
    Uninstalling pip-25.1.1:
      Successfully uninstalled pip-25.1.1


### 유틸리티 함수 정의 (시각화 및 저장)

In [ ]:
def plot_grid(images, path, grid_size, scale=2):
    fig = plt.figure(figsize=(grid_size * scale, grid_size * scale))
    plt.subplots_adjust(wspace=0, hspace=0)
    for i, img in enumerate(images):
        ax = plt.subplot(grid_size, grid_size, i + 1)
        ax.imshow(np.array(img))
        ax.axis("off")
    plt.savefig(path, bbox_inches="tight", pad_inches=0)
    plt.show()

def export_as_gif(filename, images, frames_per_second=10):
    images[0].save(
        filename, save_all=True, append_images=images[1:], 
        duration=1000 // frames_per_second, loop=0
    )

### 4개 프롬프트 기반 2D 잠재 공간 보간

In [ ]:
# 1. 텍스트 인코딩 함수
def get_encoding(prompt):
    inputs = pipe.tokenizer(prompt, padding="max_length", max_length=pipe.tokenizer.model_max_length, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        encoding = pipe.text_encoder(**inputs).last_hidden_state
    return encoding.squeeze(0)

# 2. 4개의 프롬프트 설정
prompt_1 = "A watercolor painting of a Golden Retriever at the beach"
prompt_2 = "A still life DSLR photo of a bowl of fruit"
prompt_3 = "The eiffel tower in the style of starry night"
prompt_4 = "An architectural sketch of a skyscraper"

# 3. 각 프롬프트 임베딩 생성
enc1, enc2, enc3, enc4 = [get_encoding(p) for p in [prompt_1, prompt_2, prompt_3, prompt_4]]

# 4. 2D 보간 수행 (6x6 그리드)
interpolation_steps = 6
alphas = torch.linspace(0, 1, interpolation_steps, device=device)
betas = torch.linspace(0, 1, interpolation_steps, device=device)

interpolated_encodings = []
for beta in betas:
    line_start = (1 - beta) * enc1 + beta * enc3
    line_end = (1 - beta) * enc2 + beta * enc4
    for alpha in alphas:
        interpolated_encodings.append((1 - alpha) * line_start + alpha * line_end)

interpolated_encodings = torch.stack(interpolated_encodings)

# 5. 이미지 생성 (고정된 노이즈 사용으로 변화 관찰)
generator = torch.Generator(device).manual_seed(12345)
latents = torch.randn((1, 4, 64, 64), generator=generator, device=device, dtype=torch_dtype).repeat(len(interpolated_encodings), 1, 1, 1)

images = []
batch_size = 4
for i in range(0, len(interpolated_encodings), batch_size):
    batch_enc = interpolated_encodings[i : i+batch_size]
    batch_lat = latents[i : i+batch_size]
    output = pipe(prompt_embeds=batch_enc, latents=batch_lat, num_inference_steps=25)
    images.extend(output.images)

plot_grid(images, "4-way-interpolation.jpg", interpolation_steps)

### Optional: 노이즈 공간의 원형 워크

In [ ]:
walk_steps = 60
angles = torch.linspace(0, 2 * math.pi, walk_steps)

# 두 개의 독립적인 노이즈 생성
noise_a = torch.randn((1, 4, 64, 64), device=device, dtype=torch_dtype)
noise_b = torch.randn((1, 4, 64, 64), device=device, dtype=torch_dtype)

walk_images = []
for angle in angles:
    # 원형 보간 수식: cos(theta)*A + sin(theta)*B
    interpolated_noise = math.cos(angle) * noise_a + math.sin(angle) * noise_b
    img = pipe(prompt=prompt_1, latents=interpolated_noise, num_inference_steps=20).images[0]
    walk_images.append(img)

export_as_gif("noise_walk.gif", walk_images)

### Dreambooth 및 LoRA 실습 세팅

In [ ]:
# 1. Dreambooth 학습용 데이터 구조 (가이드)
# instance_prompt: "a photo of sks dog"
# instance_data_dir: 강아지 사진 5-6장 폴더 경로

# 2. LoRA 및 체크포인트 적용
custom_model = "digiplay/hellofantasytime_v1.22"
pipe_custom = StableDiffusionPipeline.from_pretrained(custom_model, torch_dtype=torch_dtype).to(device)

# 가중치 로드 (실제 파일 경로 확인 필요)
# pipe_custom.load_lora_weights("./lora_weight.safetensors")

# 생성 테스트
final_img = pipe_custom(
    prompt="masterpiece, best quality, sks dog in the style of fantasy",
    guidance_scale=7.5, # 이 값이 높을수록 프롬프트에 충실함
    num_inference_steps=30
).images[0]
final_img.show()